# Optional workflow: custom aSoCC inter-method weights


The template function writes `equal_weights.csv`, `README_inter_method_weights.txt`, and the rendered
probability tree for one allocated shares of carrying capacities (aSoCC) method scope.
The preview function validates an edited custom inter-method tree and renders its
preview figure.


## Workspace setup

Run `set_workspace(...)` before any later function call. It sets the active workspace root
for the current Python session, creates or reuses the workspace, imports package
prerequisites under `data_raw/`, and records setup guidance in `data_raw/summary.log`.

In [ ]:
from pyaesa import set_workspace

# Windows example; update this path before running.
set_workspace(r"C:\Users\username\Documents\aesa_workspace")
# macOS example; update this path before running.
# set_workspace("/Users/username/Documents/aesa_workspace")

Methodological notes from the repository methodological notes folder are imported with package
prerequisites:

* `data_raw/methodological_notes/` contains notes for definition
  of functional units and allocation methods, prospective allocation, and
  uncertainty sources.
* `data_raw/carrying_capacities/` contains the note for definition of steady
  state and dynamic carrying capacities.


## Functional units and selectors

Use [tutorials/study_objectives/1_functional_units_and_allocation_methods.md](../study_objectives/1_functional_units_and_allocation_methods.md) to choose the FU and required selectors before building the inter-method tree. Use <a href="../../methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf">methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</a> for detailed FU definitions and mathematical expressions.


## aSoCC method scope


The custom weight functions use the same aSoCC method scope as deterministic and uncertainty aSoCC. Use [tutorials/study_objectives/1_functional_units_and_allocation_methods.md](../study_objectives/1_functional_units_and_allocation_methods.md) for available allocation methods per FU and for `method_plan`, `l1_methods`, `one_step_methods`, `two_step_methods`, and `l1_l2_pairs` syntax. Use <a href="../../methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf">methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</a> for detailed method definitions and mathematical expressions.

`method_plan="default"` includes all <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span> allocation methods available for the selected FU. Use the selector arguments only when the inter-method tree should cover a smaller method scope. `lcia_method` is required when LCIA based allocation methods are included. Custom LCIA methods are prepared through `data_raw/mrio/exiobase_3/lcia/characterization_factors_matrices/README_add_custom_lcia_characterization_matrices.txt` and then processed with `process_mrio(...)`.


## Public argument checklist
The table lists all arguments; the same definitions are available in the function docstring.

<div class="pyaesa-argument-legend">
<div class="pyaesa-default-block" style="color:#087f5b"><strong>Green items = default if omitted.</strong></div>
<div class="pyaesa-optional-block" style="color:#c45f00"><strong>Orange items = optional feature skipped if omitted.</strong></div>
</div>

Do not write green or orange items when that behavior is intended.

<details open>
<summary><code>write_asocc_weight_template(...)</code> arguments</summary>

<table>
<thead><tr><th>Argument</th><th>Description</th></tr></thead>
<tbody>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>base_asocc_args</code></td><td>Deterministic aSoCC selector envelope used to resolve<br>&#10;the final method leaves represented by the tree. Write nested<br>&#10;arguments as <code>base_asocc_args={&quot;project_name&quot;: &quot;...&quot;, &quot;source&quot;: &quot;...&quot;, &quot;fu_code&quot;: &quot;...&quot;}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>project_name</code>: Required project name used to build<br>&#10;  <code>&lt;repo&gt;/&lt;project_name&gt;</code>.<br>&#10;&bull;&nbsp;<code>source</code>: MRIO source key (<code>&quot;exiobase_396_ixi&quot;</code>,<br>&#10;  <code>&quot;exiobase_396_pxp&quot;</code>, <code>&quot;exiobase_3102_ixi&quot;</code>,<br>&#10;  <code>&quot;exiobase_3102_pxp&quot;</code>, or <code>&quot;oecd_v2025&quot;</code>), or <code>&quot;iso3&quot;</code><br>&#10;  for ISO3 only mode (L1 EG/PR(GDPcap) only).<br>&#10;<span style="color:#c45f00">&bull;&nbsp;<code>agg_reg</code>: If <code>True</code>, reclassify MRIO regions with the <code>agg_reg_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping. The mapping can keep native labels, aggregate several native regions into one target label, or disaggregate one native region across several target labels when a <code>weight</code> column is provided.<br>&#10;<strong>Default</strong> <code>False</code> keeps native source regions.</span><br>&#10;<span style="color:#c45f00">&bull;&nbsp;<code>agg_sec</code>: If <code>True</code>, reclassify MRIO sectors with the <code>agg_sec_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping. The mapping can keep native labels, aggregate several native sectors into one target label, or disaggregate one native sector across several target labels when a <code>weight</code> column is provided.<br>&#10;<strong>Default</strong> <code>False</code> keeps native source sectors.</span><br>&#10;<span style="color:#c45f00">&bull;&nbsp;<code>agg_version</code>: Name token used to resolve the matching <code>agg_reg_&lt;agg_version&gt;.csv</code> and/or <code>agg_sec_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping files in <code>data_raw/mrio/&lt;source&gt;/aggregation</code>. Required when <code>agg_reg</code> or <code>agg_sec</code> is True.<br>&#10;<strong>Defaults to</strong> an empty string for native source classification. Use the same token in downstream calls that should reuse the processed classification. If a mapping file has a <code>weight</code> column, weights must sum to <code>1</code> for each original label.</span><br>&#10;&bull;&nbsp;<code>years</code>: Studied years. Accepts a single year, list, or<br>&#10;  range. <strong>If omitted</strong>, all available MRIO years for the selected<br>&#10;  source and <code>agg_version</code> are used.<br>&#10;&bull;&nbsp;<code>fu_code</code>: Required functional unit code (for example<br>&#10;  <code>&quot;L1.a&quot;</code>, <code>&quot;L2.c.b&quot;</code>). See<br>&#10;  <code>data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</code><br>&#10;  for all available functional unit codes and the system<br>&#10;  boundaries each represents.<br>&#10;&bull;&nbsp;<code>s_p</code>: Producing sector filter(s), single string or list. If<br>&#10;  this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid producing sectors. To<br>&#10;  identify valid sector names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../aggregation/.../agg_sec_template.csv</code><br>&#10;  file. For EXIOBASE sector definitions, see<br>&#10;  <code>data_raw/mrio/exiobase_3/sector_classification.xlsx</code>;<br>&#10;  EXIOBASE ixi and pxp use different sector lists.<br>&#10;&bull;&nbsp;<code>r_p</code>: Producing region filter(s), single string or list. If<br>&#10;  this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid producing regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code><br>&#10;  file.<br>&#10;&bull;&nbsp;<code>r_c</code>: Consuming region filter(s), single string or list. If<br>&#10;  this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid consuming regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code><br>&#10;  file.<br>&#10;&bull;&nbsp;<code>r_f</code>: Final demand region filter(s), single string or list.<br>&#10;  If this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid final demand regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code><br>&#10;  file.<br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>group_indices</code>: Whether multiple selected region or sector filter values are kept as separate result rows or summed into one result row after the function calculation has been performed.<br>&#10;&bull;&nbsp;<code>False</code> (<strong>default</strong>): keep selected values as independent rows.<br>&#10;&bull;&nbsp;<code>True</code>: sum selected values into one result row.<br>&#10;The function refuses to run when <code>group_indices=True</code> is used with <code>L2.a.b</code>, <code>L2.b.b</code>, or <code>L2.c.b</code> because summing output rows for CBA total demand boundaries can double count. For these functional units, change the upstream MRIO aggregation and disaggregation scope with <code>agg_reg</code>, <code>agg_sec</code>, and <code>agg_version</code> before running the study.</span><br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>method_plan</code>: <code>method_plan</code> <strong>defaults to</strong> <code>&quot;default&quot;</code> and<br>&#10;  accepts <code>&quot;default&quot;</code>, <code>&quot;one_step&quot;</code>, <code>&quot;two_steps&quot;</code>,<br>&#10;  <code>&quot;pairs&quot;</code>, or <code>&quot;one_step_pairs&quot;</code>. <strong>When omitted</strong>, all <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span><br>&#10;  allocation methods available for the selected <code>fu_code</code> are<br>&#10;  applied. See<br>&#10;  <code>data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</code><br>&#10;  for the allocation methods available per functional unit,<br>&#10;  including definitions and mathematical expressions.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>l1_methods</code>: Optional L1 subset. <strong>Omit</strong> it to keep all L1<br>&#10;  methods allowed by <code>method_plan</code>. In <code>&quot;default&quot;</code>, this<br>&#10;  filters only L1 weights used by two step methods. In<br>&#10;  <code>&quot;two_steps&quot;</code>, it filters the two step cartesian L1 side.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>one_step_methods</code>: Optional one step L2 subset. <strong>Omit</strong> it to<br>&#10;  keep all one step methods allowed by <code>method_plan</code>.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>two_step_methods</code>: Optional two step L2 subset. <strong>Omit</strong> it to<br>&#10;  keep all two step L2 methods allowed by <code>method_plan</code>.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>l1_l2_pairs</code>: Explicit pair list formatted as<br>&#10;  <code>&quot;L1METHOD::L2METHOD&quot;</code>. <strong>Omit</strong> it unless <code>method_plan</code> is<br>&#10;  <code>&quot;pairs&quot;</code> or <code>&quot;one_step_pairs&quot;</code>.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>l1_reg_aggreg</code>: L1 aggregation mode for methods where timing<br>&#10;  matters (<code>PR(GDPcap)</code>, <code>PR-HR(Ecap)</code> and <code>AR(Ecap)</code>).<br>&#10;  <code>&quot;pre&quot;</code> aggregates regions before L1 computation. <code>&quot;post&quot;</code><br>&#10;  (<strong>default</strong>) computes on original regions, then aggregates.</span><br>&#10;&bull;&nbsp;<code>lcia_method</code>: LCIA method(s) to characterize for EXIO<br>&#10;  processing (for example <code>&quot;pb_lcia&quot;</code> or<br>&#10;  <code>[&quot;pb_lcia&quot;, &quot;gwp100_lcia&quot;]</code>). <code>None</code> means LCIA<br>&#10;  characterization is skipped. <strong>Defaults to</strong> <code>None</code>. LCIA<br>&#10;  characterization is available only for EXIOBASE sources. To add<br>&#10;  a custom LCIA method, follow<br>&#10;  <code>README_add_custom_lcia_characterization_matrices.txt</code> in<br>&#10;  <code>data_raw/mrio/exiobase_3/lcia/characterization_factors_matrices/</code><br>&#10;  and pass the method file stem here.<br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>reference_years</code>: Acquired rights (AR) methods reference<br>&#10;  year selector. Accepts a single year, list, or range. If<br>&#10;  <strong>omitted</strong>, AR routes use all historical years in the studied range<br>&#10;  up to the source registry historical cutoff. For EXIOBASE<br>&#10;  3.10.2 and OECD ICIO v2025, the cutoff is 2022; other supported<br>&#10;  MRIO sources use their own registry cutoff.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>ssp_scenario</code>: SSP scenario name or list. <strong>Defaults to</strong><br>&#10;  <code>[&quot;SSP1&quot;, &quot;SSP2&quot;, &quot;SSP3&quot;, &quot;SSP4&quot;, &quot;SSP5&quot;]</code> and is applied<br>&#10;  when scenario dependent inputs are required.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>projection_mode</code>: Projection policy for post historical<br>&#10;  years of L2 utilitarian (UT) methods (MRIO economic enacting<br>&#10;  metrics). <strong>Defaults to</strong> <code>&quot;regression&quot;</code>. <code>&quot;regression&quot;</code><br>&#10;  projects UT inputs for future years. <code>&quot;historical_reuse&quot;</code><br>&#10;  reuses historical UT structures.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>reg_window</code>: Historical regression fit window for regression<br>&#10;  mode. Provide it as <code>range(start_year, end_year + 1)</code> or as<br>&#10;  an explicit list of consecutive years in fit window order. When<br>&#10;  <strong>omitted</strong>, the source registry supplies the <strong>default</strong> fit window<br>&#10;  from the modeled year minimum through the source historical<br>&#10;  cutoff. For EXIOBASE 3.10.2 and OECD ICIO v2025, this resolves<br>&#10;  to 1995 to 2022; other supported MRIO sources use their own<br>&#10;  registry window.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>l2_reuse_years</code>: Historical L2 reuse year selector used by<br>&#10;  all UT historical reuse routes. In<br>&#10;  <code>projection_mode=&quot;historical_reuse&quot;</code> it applies to all UT<br>&#10;  methods; in <code>projection_mode=&quot;regression&quot;</code> it applies to<br>&#10;  adjusted UT routes (<code>UT(FDa)</code>, <code>UT(GVAa)</code>), which always<br>&#10;  use historical reuse as regression is not supported (would<br>&#10;  require regression on full MRIO). <strong>If omitted</strong>, <strong>defaults to</strong><br>&#10;  <code>reg_window</code> when required.</span></td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>external_method</code></td><td>Optional external aSoCC method selector. Use<br>&#10;<code>{&quot;l1_methods&quot;: [...]}</code> for L1 functional units. For L2<br>&#10;functional units use <code>{&quot;one_step_methods&quot;: [...]}</code> and/or<br>&#10;<code>{&quot;l1_l2_pairs&quot;: [&quot;&lt;l1_method&gt;::&lt;l2_method&gt;&quot;, ...]}</code>.<br>&#10;<strong>Omit</strong> this argument or pass <code>None</code> when using only native <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span><br>&#10;deterministic aSoCC methods. Use <code>prepare_external_inputs(...)</code><br>&#10;to import the external aSoCC runnable CSV examples and <code>README_external_asocc_templates.txt</code>, then<br>&#10;follow that README for external method names, selector<br>&#10;syntax, and deterministic or Monte Carlo external aSoCC CSV<br>&#10;preparation.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_format</code></td><td>Figure render settings mapping. <strong>Defaults to</strong><br>&#10;<code>{&quot;format&quot;: &quot;png&quot;, &quot;dpi&quot;: 500}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>format</code>: Figure file format. Accepted values are <code>&quot;png&quot;</code>,<br>&#10;  <code>&quot;pdf&quot;</code>, and <code>&quot;svg&quot;</code>.<br>&#10;&bull;&nbsp;<code>dpi</code>: Positive integer figure resolution used for raster<br>&#10;  outputs.</td></tr>
</tbody>
</table>

</details>

<details open>
<summary><code>preview_asocc_weight_tree(...)</code> arguments</summary>

<table>
<thead><tr><th>Argument</th><th>Description</th></tr></thead>
<tbody>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>base_asocc_args</code></td><td>Deterministic aSoCC selector envelope used to rebuild<br>&#10;the expected final method tree. Write nested arguments as<br>&#10;<code>base_asocc_args={&quot;project_name&quot;: &quot;...&quot;, &quot;source&quot;: &quot;...&quot;, &quot;fu_code&quot;: &quot;...&quot;}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>project_name</code>: Required project name used to build<br>&#10;  <code>&lt;repo&gt;/&lt;project_name&gt;</code>.<br>&#10;&bull;&nbsp;<code>source</code>: MRIO source key (<code>&quot;exiobase_396_ixi&quot;</code>,<br>&#10;  <code>&quot;exiobase_396_pxp&quot;</code>, <code>&quot;exiobase_3102_ixi&quot;</code>,<br>&#10;  <code>&quot;exiobase_3102_pxp&quot;</code>, or <code>&quot;oecd_v2025&quot;</code>), or <code>&quot;iso3&quot;</code><br>&#10;  for ISO3 only mode (L1 EG/PR(GDPcap) only).<br>&#10;<span style="color:#c45f00">&bull;&nbsp;<code>agg_reg</code>: If <code>True</code>, reclassify MRIO regions with the <code>agg_reg_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping. The mapping can keep native labels, aggregate several native regions into one target label, or disaggregate one native region across several target labels when a <code>weight</code> column is provided.<br>&#10;<strong>Default</strong> <code>False</code> keeps native source regions.</span><br>&#10;<span style="color:#c45f00">&bull;&nbsp;<code>agg_sec</code>: If <code>True</code>, reclassify MRIO sectors with the <code>agg_sec_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping. The mapping can keep native labels, aggregate several native sectors into one target label, or disaggregate one native sector across several target labels when a <code>weight</code> column is provided.<br>&#10;<strong>Default</strong> <code>False</code> keeps native source sectors.</span><br>&#10;<span style="color:#c45f00">&bull;&nbsp;<code>agg_version</code>: Name token used to resolve the matching <code>agg_reg_&lt;agg_version&gt;.csv</code> and/or <code>agg_sec_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping files in <code>data_raw/mrio/&lt;source&gt;/aggregation</code>. Required when <code>agg_reg</code> or <code>agg_sec</code> is True.<br>&#10;<strong>Defaults to</strong> an empty string for native source classification. Use the same token in downstream calls that should reuse the processed classification. If a mapping file has a <code>weight</code> column, weights must sum to <code>1</code> for each original label.</span><br>&#10;&bull;&nbsp;<code>years</code>: Studied years. Accepts a single year, list, or<br>&#10;  range. <strong>If omitted</strong>, all available MRIO years for the selected<br>&#10;  source and <code>agg_version</code> are used.<br>&#10;&bull;&nbsp;<code>fu_code</code>: Required functional unit code (for example<br>&#10;  <code>&quot;L1.a&quot;</code>, <code>&quot;L2.c.b&quot;</code>). See<br>&#10;  <code>data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</code><br>&#10;  for all available functional unit codes and the system<br>&#10;  boundaries each represents.<br>&#10;&bull;&nbsp;<code>s_p</code>: Producing sector filter(s), single string or list. If<br>&#10;  this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid producing sectors. To<br>&#10;  identify valid sector names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../aggregation/.../agg_sec_template.csv</code><br>&#10;  file. For EXIOBASE sector definitions, see<br>&#10;  <code>data_raw/mrio/exiobase_3/sector_classification.xlsx</code>;<br>&#10;  EXIOBASE ixi and pxp use different sector lists.<br>&#10;&bull;&nbsp;<code>r_p</code>: Producing region filter(s), single string or list. If<br>&#10;  this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid producing regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code><br>&#10;  file.<br>&#10;&bull;&nbsp;<code>r_c</code>: Consuming region filter(s), single string or list. If<br>&#10;  this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid consuming regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code><br>&#10;  file.<br>&#10;&bull;&nbsp;<code>r_f</code>: Final demand region filter(s), single string or list.<br>&#10;  If this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid final demand regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code><br>&#10;  file.<br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>group_indices</code>: Whether multiple selected region or sector filter values are kept as separate result rows or summed into one result row after the function calculation has been performed.<br>&#10;&bull;&nbsp;<code>False</code> (<strong>default</strong>): keep selected values as independent rows.<br>&#10;&bull;&nbsp;<code>True</code>: sum selected values into one result row.<br>&#10;The function refuses to run when <code>group_indices=True</code> is used with <code>L2.a.b</code>, <code>L2.b.b</code>, or <code>L2.c.b</code> because summing output rows for CBA total demand boundaries can double count. For these functional units, change the upstream MRIO aggregation and disaggregation scope with <code>agg_reg</code>, <code>agg_sec</code>, and <code>agg_version</code> before running the study.</span><br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>method_plan</code>: <code>method_plan</code> <strong>defaults to</strong> <code>&quot;default&quot;</code> and<br>&#10;  accepts <code>&quot;default&quot;</code>, <code>&quot;one_step&quot;</code>, <code>&quot;two_steps&quot;</code>,<br>&#10;  <code>&quot;pairs&quot;</code>, or <code>&quot;one_step_pairs&quot;</code>. <strong>When omitted</strong>, all <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span><br>&#10;  allocation methods available for the selected <code>fu_code</code> are<br>&#10;  applied. See<br>&#10;  <code>data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</code><br>&#10;  for the allocation methods available per functional unit,<br>&#10;  including definitions and mathematical expressions.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>l1_methods</code>: Optional L1 subset. <strong>Omit</strong> it to keep all L1<br>&#10;  methods allowed by <code>method_plan</code>. In <code>&quot;default&quot;</code>, this<br>&#10;  filters only L1 weights used by two step methods. In<br>&#10;  <code>&quot;two_steps&quot;</code>, it filters the two step cartesian L1 side.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>one_step_methods</code>: Optional one step L2 subset. <strong>Omit</strong> it to<br>&#10;  keep all one step methods allowed by <code>method_plan</code>.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>two_step_methods</code>: Optional two step L2 subset. <strong>Omit</strong> it to<br>&#10;  keep all two step L2 methods allowed by <code>method_plan</code>.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>l1_l2_pairs</code>: Explicit pair list formatted as<br>&#10;  <code>&quot;L1METHOD::L2METHOD&quot;</code>. <strong>Omit</strong> it unless <code>method_plan</code> is<br>&#10;  <code>&quot;pairs&quot;</code> or <code>&quot;one_step_pairs&quot;</code>.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>l1_reg_aggreg</code>: L1 aggregation mode for methods where timing<br>&#10;  matters (<code>PR(GDPcap)</code>, <code>PR-HR(Ecap)</code> and <code>AR(Ecap)</code>).<br>&#10;  <code>&quot;pre&quot;</code> aggregates regions before L1 computation. <code>&quot;post&quot;</code><br>&#10;  (<strong>default</strong>) computes on original regions, then aggregates.</span><br>&#10;&bull;&nbsp;<code>lcia_method</code>: LCIA method(s) to characterize for EXIO<br>&#10;  processing (for example <code>&quot;pb_lcia&quot;</code> or<br>&#10;  <code>[&quot;pb_lcia&quot;, &quot;gwp100_lcia&quot;]</code>). <code>None</code> means LCIA<br>&#10;  characterization is skipped. <strong>Defaults to</strong> <code>None</code>. LCIA<br>&#10;  characterization is available only for EXIOBASE sources. To add<br>&#10;  a custom LCIA method, follow<br>&#10;  <code>README_add_custom_lcia_characterization_matrices.txt</code> in<br>&#10;  <code>data_raw/mrio/exiobase_3/lcia/characterization_factors_matrices/</code><br>&#10;  and pass the method file stem here.<br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>reference_years</code>: Acquired rights (AR) methods reference<br>&#10;  year selector. Accepts a single year, list, or range. If<br>&#10;  <strong>omitted</strong>, AR routes use all historical years in the studied range<br>&#10;  up to the source registry historical cutoff. For EXIOBASE<br>&#10;  3.10.2 and OECD ICIO v2025, the cutoff is 2022; other supported<br>&#10;  MRIO sources use their own registry cutoff.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>ssp_scenario</code>: SSP scenario name or list. <strong>Defaults to</strong><br>&#10;  <code>[&quot;SSP1&quot;, &quot;SSP2&quot;, &quot;SSP3&quot;, &quot;SSP4&quot;, &quot;SSP5&quot;]</code> and is applied<br>&#10;  when scenario dependent inputs are required.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>projection_mode</code>: Projection policy for post historical<br>&#10;  years of L2 utilitarian (UT) methods (MRIO economic enacting<br>&#10;  metrics). <strong>Defaults to</strong> <code>&quot;regression&quot;</code>. <code>&quot;regression&quot;</code><br>&#10;  projects UT inputs for future years. <code>&quot;historical_reuse&quot;</code><br>&#10;  reuses historical UT structures.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>reg_window</code>: Historical regression fit window for regression<br>&#10;  mode. Provide it as <code>range(start_year, end_year + 1)</code> or as<br>&#10;  an explicit list of consecutive years in fit window order. When<br>&#10;  <strong>omitted</strong>, the source registry supplies the <strong>default</strong> fit window<br>&#10;  from the modeled year minimum through the source historical<br>&#10;  cutoff. For EXIOBASE 3.10.2 and OECD ICIO v2025, this resolves<br>&#10;  to 1995 to 2022; other supported MRIO sources use their own<br>&#10;  registry window.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>l2_reuse_years</code>: Historical L2 reuse year selector used by<br>&#10;  all UT historical reuse routes. In<br>&#10;  <code>projection_mode=&quot;historical_reuse&quot;</code> it applies to all UT<br>&#10;  methods; in <code>projection_mode=&quot;regression&quot;</code> it applies to<br>&#10;  adjusted UT routes (<code>UT(FDa)</code>, <code>UT(GVAa)</code>), which always<br>&#10;  use historical reuse as regression is not supported (would<br>&#10;  require regression on full MRIO). <strong>If omitted</strong>, <strong>defaults to</strong><br>&#10;  <code>reg_window</code> when required.</span></td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>version_name</code></td><td>Required custom version token. The function reads<br>&#10;<code>preview_inter_method_weights/weights__&lt;version_name&gt;.csv</code>.<br>&#10;The token must be non empty, contain only letters, digits, and<br>&#10;underscores. The reserved token <code>&quot;equal_weight_default&quot;</code> is<br>&#10;excluded.</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>external_method</code></td><td>Optional external aSoCC method selector. Use<br>&#10;<code>{&quot;l1_methods&quot;: [...]}</code> for L1 functional units. For L2<br>&#10;functional units use <code>{&quot;one_step_methods&quot;: [...]}</code> and/or<br>&#10;<code>{&quot;l1_l2_pairs&quot;: [&quot;&lt;l1_method&gt;::&lt;l2_method&gt;&quot;, ...]}</code>.<br>&#10;<strong>Omit</strong> this argument or pass <code>None</code> when using only native <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span><br>&#10;deterministic aSoCC methods. Use <code>prepare_external_inputs(...)</code><br>&#10;to import the external aSoCC runnable CSV examples and <code>README_external_asocc_templates.txt</code>, then<br>&#10;follow that README for external method names, selector<br>&#10;syntax, and deterministic or Monte Carlo external aSoCC CSV<br>&#10;preparation.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_format</code></td><td>Figure render settings mapping. <strong>Defaults to</strong><br>&#10;<code>{&quot;format&quot;: &quot;png&quot;, &quot;dpi&quot;: 500}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>format</code>: Figure file format. Accepted values are <code>&quot;png&quot;</code>,<br>&#10;  <code>&quot;pdf&quot;</code>, and <code>&quot;svg&quot;</code>.<br>&#10;&bull;&nbsp;<code>dpi</code>: Positive integer figure resolution used for raster<br>&#10;  outputs.</td></tr>
</tbody>
</table>

</details>


## Write the editable equal weight template, using defaults where omitted


In [ ]:
from pyaesa import write_asocc_weight_template

write_asocc_weight_template(
    base_asocc_args={
        "project_name": "paper_fr_demo",
        "source": "exiobase_3102_ixi",
        "years": 2024,
        "fu_code": "L2.c.b",
        "s_p": ["Paper"],
        "r_c": ["FR"],
        "lcia_method": "gwp100_lcia",
    },
)

## Preview an edited custom weight version, using defaults where omitted


In [ ]:
from pyaesa import preview_asocc_weight_tree

preview_asocc_weight_tree(
    base_asocc_args={
        "project_name": "paper_fr_demo",
        "source": "exiobase_3102_ixi",
        "years": 2024,
        "fu_code": "L2.c.b",
        "s_p": ["Paper"],
        "r_c": ["FR"],
        "lcia_method": "gwp100_lcia",
    },
    version_name="study_weights_v1",
)